# PARCO for the PVRPWDP

In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch
from rl4co.utils.trainer import RL4COTrainer

from parco.envs.pvrpwdp import PVRPWDPVEnv
from parco.envs.pvrpwdp.generator import PVRPWDPGenerator
from parco.models import PARCORLModule, PARCOPolicy

# Ensure we're in the project root directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

# Device setup - auto-detect CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Verify data files exist
print(f"\nData files check:")
print(f"  Training epoch 0: {os.path.exists('data/train_data_npz/pvrpwdp_epoch_00.npz')}")
print(f"  Validation: {os.path.exists('data/val_data/val.npz')}")
print(f"  Test: {os.path.exists('data/test_data/test.npz')}")

e:\Đồ án\parco\.venv\Lib\site-packages\torchrl\data\replay_buffers\samplers.py:37: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)
e:\Đồ án\parco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: e:\Đồ án\parco
Using device: cuda
CUDA available: True

Data files check:
  Training epoch 0: True
  Validation: True
  Test: True


## Environment Setup


In [2]:
# Environment configuration
import os

# Get absolute paths to data directories
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
epoch_data_dir = os.path.join(project_root, "data", "train_data_npz")
val_file = os.path.join("val_data", "val.npz")  # RL4CO adds 'data/' prefix
test_file = os.path.join("test_data", "test.npz")  # RL4CO adds 'data/' prefix

env = PVRPWDPVEnv(
    # Epoch data configuration (for training) - USE ABSOLUTE PATH
    epoch_data_dir=epoch_data_dir,
    epoch_file_pattern="pvrpwdp_epoch_{epoch:02d}.npz",
    use_epoch_data=True,
    fallback_to_generator=False,  # ❌ DISABLE FALLBACK - ONLY USE NPZ FILES!
    
    # Standard validation/test files (RL4CO adds 'data/' prefix automatically)
    val_file=val_file,
    test_file=test_file,
)

# Print epoch data information
env.print_epoch_data_info()


EPOCH DATA CONFIGURATION
Epoch Data Directory: e:\Đồ án\parco\data\train_data_npz
File Pattern:         pvrpwdp_epoch_{epoch:02d}.npz
Use Epoch Data:       True
Fallback to Generator: False
Current Epoch:        0
Max Epochs:           None

Available Epochs:     0



In [3]:
emb_dim = 128

# Policy: Neural network for decision making
policy = PARCOPolicy(
    env_name=env.name,
    embed_dim=emb_dim,
    agent_handler="highprob",
    normalization="rms",
    context_embedding_kwargs={
        "normalization": "rms",
        "norm_after": False,
        "normalize_endurance_by_max": False,  # Normalize by time_scaler for consistency
        "use_time_to_depot": True,            # Include time-to-depot and time-to-deadline
    },
    norm_after=False,
)

# RL Module: Policy + Environment + Training algorithm
model = PARCORLModule(
    env, 
    policy=policy,
    train_data_size=100352,       # Instances per epoch (overridden by epoch files)
    val_data_size=2000,          # Validation set size
    batch_size=32,              # Training batch size
    num_augment=16,              # SymNCO augmentations for shared baseline
    train_min_agents=4,         # For static mode (not used with dynamic_resample)
    train_max_agents=4,         # For static mode (not used with dynamic_resample)
    train_min_size=20,          # For static mode (not used with dynamic_resample)
    train_max_size=20,          # For static mode (not used with dynamic_resample)
    use_padding_mode=True,
    optimizer_kwargs={
        'lr': 1e-4,             # Learning rate
        'weight_decay': 0
    },
)

print("✅ Model created successfully with DYNAMIC RESAMPLE enabled!")
print(f"   Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")
print(f"   Resample strategy: DYNAMIC (num_locs from NPZ → num_agents determined by size)")
print(f"   - num_locs ≤ 40: agents 4-6")
print(f"   - 40 < num_locs ≤ 80: agents 6-10")
print(f"   - num_locs > 80: agents 8-12")

✅ Model created successfully with DYNAMIC RESAMPLE enabled!
   Policy parameters: 991,489
   Resample strategy: DYNAMIC (num_locs from NPZ → num_agents determined by size)
   - num_locs ≤ 40: agents 4-6
   - 40 < num_locs ≤ 80: agents 6-10
   - num_locs > 80: agents 8-12


e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


## Training


In [4]:
trainer = RL4COTrainer(
    max_epochs=1,           # Number of training epochs (matching available data)
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,               # Number of devices
    logger=None,             # Set to wandb/tensorboard for logging
    gradient_clip_val=1.0,   # Gradient clipping
)

print("🚀 Starting training...")
print(f"   Max epochs: {trainer.max_epochs}")
print(f"   Accelerator: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


🚀 Starting training...
   Max epochs: 1
   Accelerator: GPU (NVIDIA GeForce RTX 3050 Laptop GPU)


e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [5]:
# Start training
trainer.fit(model)

print("\n✅ Training completed!")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type        | Params | Mode 
--------------------------------------------------------
0 | env             | PVRPWDPVEnv | 0      | train
1 | policy          | PARCOPolicy | 991 K  | train
2 | baseline        | NoBaseline  | 0      | train
3 | projection_head | MLP         | 33.0 K | train
--------------------------------------------------------
1.0 M     Trainable params
0         Non-trainable params
1.0 M     Total params
4.098     Total estimated model params size (MB)
91        Modules in train mode
0         Modules in eval mode


Sanity Checking:   0%|          | 0/2 [00:00<?, ?it/s]

e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

e:\Đồ án\parco\.venv\Lib\site-packages\torch\nn\functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = struct c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\layer_norm.cpp:347.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0:  37%|███▋      | 1166/3136 [1:18:13<2:12:09,  0.25it/s, v_num=1, train/reward=-0.152, train/loss=0.0689]   


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:

# Plot training rewards
import matplotlib.pyplot as plt

print("\n📊 Plotting Training Rewards...")

# Get metrics from trainer
if hasattr(trainer, 'logged_metrics'):
    rewards = trainer.logged_metrics.get('train_reward', [])
    if rewards:
        plt.figure(figsize=(10, 6))
        plt.plot(rewards, marker='o', linestyle='-', linewidth=2, markersize=6)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Reward', fontsize=12)
        plt.title('Training Reward Over Epochs', fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"✅ Reward plot displayed!")
        print(f"   Final reward: {rewards[-1]:.4f}")
    else:
        print("⚠️  No reward metrics found in trainer logs")
else:
    print("⚠️  No logged metrics available from trainer")
